<a href="https://colab.research.google.com/github/DmitryMatsykin/Python/blob/main/%D0%A3%D1%80%D0%BE%D0%BA_5_SQL_%D0%B8_Python_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background-color:#DFC7F2; padding:15px; margin:15px;">

### Урок 5. SQL и Python: подключение к базе, запросы и визуализация


## Часть 1. SQLAlchemy и драйверы баз данных

Прежде чем писать код, надо понять, из чего вообще состоит связка «Python и база данных». Здесь два слоя, и их часто путают.

<div style="background-color: rgba(14, 86, 135, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(43, 171, 226, 0.57);">

✍️ **_Драйвер (DBAPI)_** — библиотека, которая физически умеет разговаривать с конкретной СУБД по её сетевому протоколу. Для PostgreSQL это, например, `psycopg2`.

✍️ **_SQLAlchemy_** — надстройка над драйвером. Сама она к базе не подключается: она умеет строить SQL, управлять соединениями и транзакциями, а «провод» до базы просит у драйвера.

Отсюда правило: **SQLAlchemy без драйвера бесполезна**. Ставить нужно обе библиотеки.

Единственное исключение — SQLite: он работает «из коробки» просто потому, что модуль `sqlite3` входит в стандартную библиотеку Python и ставить ничего не надо. Для всех остальных баз драйвер устанавливается отдельно.

<div style="background-color: rgba(244, 194, 194, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(244, 194, 194, 0.57);">

**Драйверы для популярных СУБД**

| СУБД | Драйверы |
| --- | --- |
| MySQL | PyMySQL, MySQL-Connector, CyMySQL, MySQL-Python |
| PostgreSQL | **psycopg2**, pg8000 |
| Microsoft SQL Server | PyODBC, pymssql |
| Oracle | cx-Oracle |
| Firebird | fdb, kinterbasdb |

Установка обычно выглядит так:
```
pip install sqlalchemy psycopg2

## Часть 2. Подключаемся к базе

Подключение состоит из трёх шагов: собрать параметры доступа, склеить из них строку подключения и создать по ней «движок».

<div style="background-color: rgba(14, 86, 135, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(43, 171, 226, 0.57);">

✍️ **_Engine_** («движок») — центральный объект SQLAlchemy. Важно понимать, чем он **не** является: это не соединение с базой.

Engine — это **фабрика соединений и пул**. Он держит наготове несколько открытых соединений (по умолчанию 5 постоянных и до 10 сверх лимита) и выдаёт их по запросу, а потом забирает обратно. Открывать соединение заново на каждый запрос дорого — пул избавляет от этих расходов.

Создаётся Engine **один раз на всю программу** и дальше переиспользуется.

In [ ]:
!pip install sqlalchemy psycopg2

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------- ----- 2.4/2.8 MB 17.8 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 15.2 MB/s  0:00:00


In [ ]:
# --- Библиотеки ---
import sqlalchemy                              # ядро SQLAlchemy (здесь нужно только для проверки версии)
import pandas as pd                            # таблицы DataFrame
from sqlalchemy import create_engine, text     # create_engine — движок БД; text — обёртка для сырого SQL
import psycopg2                                # драйвер PostgreSQL; SQLAlchemy вызывает его сама

#sqlalchemy.__version__                        # синтаксис версий 1.x и 2.x различается — версию полезно знать

# Параметры подключения
connection = {
    'host': 'rc1b-7ng6ih3jte3824x8.mdb.yandexcloud.net',   # хост Managed PostgreSQL в Яндекс.Облаке
    'port': 6432,                                          # 6432 — порт пулера соединений (не стандартный 5432!)
    'database': 'demo',                                    # имя базы данных
    'user': 'root',                                        # пользователь
    'password': 'password!!!',                             # ⚠️ в реальном проекте пароль в коде не хранят
}


url = ('postgresql://{user}:{password}@{host}:{port}/{database}'
           .format(**connection))

# Создаём движок. Вызов ЛЕНИВЫЙ: прямо сейчас к базе никто не подключается,
# первое реальное соединение откроется при первом запросе.
engine = create_engine(url)

<div style="background-color: rgba(244, 194, 194, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(244, 194, 194, 0.57);">

**Как устроена строка подключения**

```
диалект+драйвер://пользователь:пароль@хост:порт/база
```

В нашем случае драйвер не указан — написано просто `postgresql://`. Тогда SQLAlchemy берёт драйвер по умолчанию, а для PostgreSQL это как раз `psycopg2`. Можно указать явно: `postgresql+psycopg2://`.


Проверим, что драйвер на месте, и заодно посмотрим его версию.

In [ ]:
print(psycopg2.__version__)   # номер версии + флаги сборки в скобках

2.9.12 (dt dec pq3 ext lo64)


В скобках после номера версии перечислены флаги сборки — по ним видно, что драйвер умеет:

* `dt` — конвертирует даты и время в питоновские объекты `datetime`;
* `dec` — числа типа `numeric` возвращает как `Decimal` (без потери точности);
* `pq3` — использует протокол libpq версии 3;
* `ext` — собраны C-расширения, то есть это быстрая версия, а не чистый Python;
* `lo64` — поддержка 64-битных больших объектов.

## Часть 3. Первый запрос

<div style="background-color: rgba(14, 86, 135, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(43, 171, 226, 0.57);">

✍️ **_Connection_** — конкретное соединение, взятое из пула движка. Именно через него выполняются запросы.

✍️ **_text()_** — обёртка, превращающая обычную строку в объект `TextClause`, который SQLAlchemy готова выполнить.

В SQLAlchemy 2.0 передать в `execute()` голую строку **нельзя** — будет ошибка `ObjectNotExecutableError`. Это сделано намеренно: разработчик должен осознанно помечать места, где он пишет сырой SQL.

In [ ]:
# with гарантирует, что соединение вернётся в пул — даже если внутри произойдёт ошибка
with engine.connect() as conn:
    # text() обязателен: голую строку execute() в SQLAlchemy 2.0 не принимает
    # lesson_1.titles — таблица titles внутри схемы lesson_1
    result = conn.execute(text("select * from lesson_1.titles"))

    conn.commit()   # транзакция открылась неявно при первом запросе — закрываем её явно

Теперь достанем сами строки.

In [ ]:
result.fetchall()   # список объектов Row — все строки результата

[(1, '1+1', 2011, 'Франция', 'Фильмы', 'Драма', 4.41),
 (2, 'Один дома', 1990, 'США', 'Фильмы', 'Комедия', 4.14),
 (3, 'Аватар', 2009, 'США', 'Фильмы', 'Фантастика', 3.99),
 (4, 'Побег из Шоушенка', 1994, 'США', 'Фильмы', 'Драма', 4.55),
 (5, 'Зверополис', 2016, 'США', 'Мультфильмы', 'Комедия', 4.16),
 (7, 'Игра престолов', 2011, 'США', 'Сериалы', 'Фэнтези', 4.49)]

`Row` печатается как обычный кортеж

In [ ]:
result.keys()   # имена колонок; работают даже после того, как строки уже вычитаны

RMKeyView(['id', 'title', 'year', 'country', 'category', 'main_genre', 'value'])

`RMKeyView` — это «вид» на список колонок, ведёт себя как `dict.keys()`: можно итерировать, обернуть в `list()`, проверить вхождение через `in`.

Метод работает даже после `fetchall()` потому, что метаданные о колонках хранятся отдельно от самих строк.

## Часть 4. Из базы сразу в DataFrame

Работать с кортежами неудобно — почти всегда результат хочется видеть таблицей pandas. Напишем небольшую функцию-обёртку, чтобы не повторять один и тот же код при каждом запросе.

In [ ]:
def get_from_bd(sql):
    with engine.connect() as conn:                  # берём соединение из пула
        query = conn.execute(text(sql))             # query — это результат (CursorResult), а не сам запрос

        df = pd.DataFrame(query.fetchall(),             # данные — список кортежей
                      columns=query.keys())         # имена колонок передаём отдельно
    return df

Проверим функцию на той же таблице.

In [ ]:
# Тройные кавычки — многострочная строка. Для SQL удобно: сохраняется форматирование запроса
lesson1 = get_from_bd('''
select *
from lesson_1.titles
''')

lesson1   # последнее выражение ячейки Jupyter выводит сам; DataFrame рисуется как HTML-таблица

,id,title,year,country,category,main_genre,value
0,1,1+1,2011,Франция,Фильмы,Драма,4.41
1,2,Один дома,1990,США,Фильмы,Комедия,4.14
2,3,Аватар,2009,США,Фильмы,Фантастика,3.99
3,4,Побег из Шоушенка,1994,США,Фильмы,Драма,4.55
4,5,Зверополис,2016,США,Мультфильмы,Комедия,4.16
5,7,Игра престолов,2011,США,Сериалы,Фэнтези,4.49


Тот же результат, что и раньше, но теперь это полноценная таблица pandas: к ней применимы `.head()`, `.describe()`, фильтрация, группировки.

Полученную таблицу можно сразу сохранить в файл.

In [ ]:
lesson1.to_csv('lesson_csv.csv')       # ⚠️ по умолчанию в файл пишется ещё и индекс
lesson1.to_excel('lesson_xlsx.xlsx')   # требует установленного openpyxl

<div style="background-color: rgba(244, 194, 194, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(244, 194, 194, 0.57);">

**Аргументы, которые почти всегда нужны**

```python
lesson1.to_csv('lesson_csv.csv',
               index=False,            # не писать индекс отдельной безымянной колонкой
               encoding='utf-8-sig')   # чтобы Excel не превратил кириллицу в кракозябры

lesson1.to_excel('lesson_xlsx.xlsx',
                 index=False,
                 sheet_name='titles')  # осмысленное имя листа вместо Sheet1
```

Без `index=False` в файле появится лишняя первая колонка с номерами строк `0, 1, 2...`.

Файлы сохраняются в **текущую рабочую директорию**.

## Часть 5. Рефлексия: база данных как объекты Python

До сих пор мы писали SQL строкой. Есть другой путь: попросить SQLAlchemy самой прочитать структуру базы и превратить таблицы в объекты Python. Тогда запросы можно собирать методами — с автодополнением в редакторе и проверкой имён колонок ещё до отправки запроса.

In [ ]:
from sqlalchemy.schema import MetaData

# MetaData — реестр объектов БД в памяти Python
metadata_demo_db = MetaData(schema = 'booking')   # схема по умолчанию для всех таблиц этого реестра

# reflect() идёт в базу и сам создаёт объекты Table для всех таблиц схемы booking
metadata_demo_db.reflect(bind = engine);          # ; в конце — привычка гасить вывод в Jupyter
                                                  # (здесь бессмысленна: reflect() и так возвращает None)

C:\Users\Asus\AppData\Local\Temp\ipykernel_10148\4238846932.py:7: SAWarning: Did not recognize type 'point' of column 'coordinates'
  metadata_demo_db.reflect(bind = engine);          # ; в конце — привычка гасить вывод в Jupyter


В выводе появилось предупреждение — не ошибка:

```
SAWarning: Did not recognize type 'point' of column 'coordinates'
```

В таблице `airports_data` колонка `coordinates` имеет геометрический тип PostgreSQL `point`, для которого у SQLAlchemy нет встроенного соответствия. Колонка всё равно создастся, но с типом-заглушкой `NullType`: работать с ней как с типизированной не выйдет. Если геоданные действительно нужны, ставят расширение **GeoAlchemy2**. На остальные таблицы это никак не влияет.

Посмотрим, что нашлось в схеме.

In [ ]:
# metadata.tables — словарь вида {'схема.таблица': объект Table}
# итерация по словарю даёт его КЛЮЧИ, поэтому печатаются имена
for table_name in metadata_demo_db.tables:
    print(table_name)

booking.aircrafts_data
booking.airports_data
booking.boarding_passes
booking.ticket_flights
booking.flights
booking.tickets
booking.bookings
booking.flights_copy
booking.seats


Девять таблиц демо-базы авиаперевозок: рейсы, билеты, посадочные талоны, аэропорты, самолёты, места. Затесавшаяся среди них `flights_copy` — явно чья-то рабочая копия, оставшаяся в базе.

Если нужны не имена, а сами объекты, итерируйте по `.items()`:

```python
for name, table in metadata_demo_db.tables.items():
    print(name, len(table.columns))
```

Теперь разложим нужные таблицы по отдельным переменным — чтобы писать `aircrafts_data.c.range`, а не `metadata_demo_db.tables['booking.aircrafts_data'].c.range`.

In [ ]:
from sqlalchemy import Table
from sqlalchemy import select


boarding_passes = Table('boarding_passes', metadata_demo_db, autoload_with=engine)
ticket_flights = Table('ticket_flights', metadata_demo_db, autoload_with=engine)
flights = Table('flights', metadata_demo_db, autoload_with=engine)
aircrafts_data = Table('aircrafts_data', metadata_demo_db, autoload_with=engine)
airports_data = Table('airports_data', metadata_demo_db, autoload_with=engine)
tickets = Table('tickets', metadata_demo_db, autoload_with=engine)
bookings = Table('bookings', metadata_demo_db, autoload_with=engine)
seats = Table('seats', metadata_demo_db, autoload_with=engine)

In [ ]:
print(aircrafts_data.columns)   # .columns (короткий синоним — .c) — коллекция объектов Column

ReadOnlyColumnCollection(aircrafts_data.aircraft_code, aircrafts_data.model, aircrafts_data.range)


В таблице три колонки: код самолёта, модель и дальность полёта. Обращаться к ним можно двумя способами: `aircrafts_data.c.range` или `aircrafts_data.c['range']`.

Полезно рядом:

```python
aircrafts_data.c.range.type      # тип колонки
aircrafts_data.primary_key       # первичный ключ
aircrafts_data.foreign_keys      # внешние ключи
```

## Часть 6. Запрос как объект, а не как строка

<div style="background-color: rgba(14, 86, 135, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(43, 171, 226, 0.57);">

✍️ **_SQLAlchemy Core_** — способ строить запросы конструктором: `select(...)`, `.where(...)`, `.order_by(...)`.

Каждый метод возвращает **новый** объект запроса, поэтому вызовы можно составлять в цепочку. Готовый SQL появится только в момент отправки в базу.

Главное отличие от строки: опечатка в имени колонки будет поймана сразу, в Python, а не через полсекунды ответом от сервера.

In [ ]:
# select(таблица) → SELECT всех её колонок. Это синтаксис 2.0; в 1.x писали select([таблица])
sql_query = select(aircrafts_data).where(
    (aircrafts_data.c.range < 5000) |             # | — это OR (не «or»!); скобки обязательны
    (aircrafts_data.c.aircraft_code == 'SU9')     # == на Column не сравнивает, а СТРОИТ SQL-условие
)

print(sql_query)      # значения заменены плейсхолдерами :range_1 и :aircraft_code_1

# literal_binds=True подставит значения прямо в текст запроса
compiled_query = sql_query.compile(
    compile_kwargs={"literal_binds": True}
)
print('')             # пустая строка-разделитель между двумя выводами
print(compiled_query)

SELECT booking.aircrafts_data.aircraft_code, booking.aircrafts_data.model, booking.aircrafts_data.range 
FROM booking.aircrafts_data 
WHERE booking.aircrafts_data.range < :range_1 OR booking.aircrafts_data.aircraft_code = :aircraft_code_1

SELECT booking.aircrafts_data.aircraft_code, booking.aircrafts_data.model, booking.aircrafts_data.range 
FROM booking.aircrafts_data 
WHERE booking.aircrafts_data.range < 5000 OR booking.aircrafts_data.aircraft_code = 'SU9'


### Преимущества такого подхода

* **Безопасность** — значения автоматически становятся параметрами, инъекция невозможна.
* **Читаемость** — код похож на SQL, но работает автодополнение, а опечатка в имени колонки видна сразу.
* **Компиляция** — точный текст запроса можно посмотреть до отправки в базу.
* **Собираемость** — условия легко добавлять в цикле или по `if`, не склеивая строки:

```python
q = select(orders)
if category:
    q = q.where(orders.c.category == category)
if date_from:
    q = q.where(orders.c.order_date >= date_from)
```

In [ ]:
with engine.connect() as conn:
    # text() не нужен: sql_query — уже готовый исполняемый объект Core
    example = conn.execute(sql_query).fetchone()   # fetchone() ВНУТРИ with — правильный шаблон


example

('733', {'en': 'Boeing 737-300', 'ru': 'Боинг 737-300'}, 4200)

Результат — первая подходящая строка: `('733', {'en': 'Boeing 737-300', 'ru': 'Боинг 737-300'}, 4200)`.

Любопытная деталь: **модель пришла питоновским словарём**. В базе эта колонка имеет тип `jsonb`, и `psycopg2` автоматически разобрал JSON. Значит, обращаться можно так:

```python
example[1]['ru']   # 'Боинг 737-300'
```

Коммита здесь нет, и он не нужен: на выходе из `with` неявная транзакция откатится, а для `SELECT` это безразлично.

## Часть 7. SQL-функции через `func`

<div style="background-color: rgba(14, 86, 135, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(43, 171, 226, 0.57);">

✍️ **_func_** — «магическая» фабрика SQL-функций. У неё нет заранее заданного списка методов: любое имя после точки превращается в вызов функции с тем же именем.

```python
func.count(orders.c.id)        →  count(orders.id)
func.coalesce(col, 0)          →  coalesce(col, 0)
func.date_trunc('month', col)  →  date_trunc('month', col)
```

Работает и с вашими собственными функциями, определёнными в базе.

In [ ]:
from sqlalchemy.sql import func   # фабрика SQL-функций

# Отдельный реестр для второй схемы — lesson_1
metadata_demo_lesson1 = MetaData(schema = 'lesson_1')
metadata_demo_lesson1.reflect(bind = engine);

orders = Table('orders', metadata_demo_lesson1, autoload=True)   # таблица заказов ресторана

Можно было обойтись одним реестром — `MetaData` спокойно держит таблицы из разных схем:

```python
metadata.reflect(bind=engine, schema='booking')
metadata.reflect(bind=engine, schema='lesson_1')
```

Теперь соберём запрос с функцией внутри условия.

In [ ]:
# ⚠️ имя sql_query переиспользуется — предыдущий запрос по самолётам затирается
sql_query = select(orders).where(
    # date_trunc обрезает дату до начала периода: 2023-05-17 → 2023-05-01
    func.date_trunc('month', orders.c.order_date) == '2023-01-01'
)

with engine.connect() as conn:
    example = conn.execute(sql_query).fetchone()


example

(1, 1, datetime.date(2023, 1, 1), 109, 'Korean Beef Bowl', 'Asian', 17.95, datetime.datetime(2023, 1, 1, 11, 38))

В ответе видна вся структура таблицы `orders`: `order_details_id`, `order_id`, `order_date`, `item_id`, `item_name`, `category`, `price`, `order_datetime`. Даты драйвер уже превратил в объекты `date` и `datetime`.

Сравнение `== '2023-01-01'` — со строкой. PostgreSQL сам приведёт её к нужному типу, но полагаться на неявные преобразования не стоит. Надёжнее сравнивать с настоящей датой: `== date(2023, 1, 1)`.

Посмотрим, во что превратился запрос.

In [ ]:
print(sql_query)   # обратите внимание: параметром стала даже константа 'month'

SELECT lesson_1.orders.order_details_id, lesson_1.orders.order_id, lesson_1.orders.order_date, lesson_1.orders.item_id, lesson_1.orders.item_name, lesson_1.orders.category, lesson_1.orders.price, lesson_1.orders.order_datetime 
FROM lesson_1.orders 
WHERE date_trunc(:date_trunc_1, lesson_1.orders.order_date) = :date_trunc_2


Показательный момент: `:date_trunc_1` — это подставленная строка `'month'`. SQLAlchemy биндит **любой** питоновский литерал, попавший в выражение, а имена плейсхолдеров генерирует сама по имени функции с порядковым номером.

In [ ]:
compiled_query = sql_query.compile(
    compile_kwargs={"literal_binds": True}
)
print(compiled_query)   # тот же запрос, но со значениями прямо в тексте — удобно копировать в pgAdmin

SELECT lesson_1.orders.order_details_id, lesson_1.orders.order_id, lesson_1.orders.order_date, lesson_1.orders.item_id, lesson_1.orders.item_name, lesson_1.orders.category, lesson_1.orders.price, lesson_1.orders.order_datetime 
FROM lesson_1.orders 
WHERE date_trunc('month', lesson_1.orders.order_date) = '2023-01-01'


## Часть 8. Собираем данные за год

Задача: получить выручку по категориям блюд за каждый месяц 2023 года. Разберём приём «цикл по периодам»: делаем запрос на каждый месяц, копим результаты в список и в конце склеиваем в одну таблицу.

Сначала подготовим список дат.

In [ ]:
from datetime import datetime
import time                      # понадобится для пауз в цикле ниже

# Месяца с января по декабрь 2023.
# Это списковое включение (list comprehension) — короткая запись цикла, собирающего список
months = [
    datetime(2023, month, 1)     # первое число месяца, время 00:00
    for month in range(1, 13)    # 1..12, правая граница НЕ включается
]
months

[datetime.datetime(2023, 1, 1, 0, 0),
 datetime.datetime(2023, 2, 1, 0, 0),
 datetime.datetime(2023, 3, 1, 0, 0),
 datetime.datetime(2023, 4, 1, 0, 0),
 datetime.datetime(2023, 5, 1, 0, 0),
 datetime.datetime(2023, 6, 1, 0, 0),
 datetime.datetime(2023, 7, 1, 0, 0),
 datetime.datetime(2023, 8, 1, 0, 0),
 datetime.datetime(2023, 9, 1, 0, 0),
 datetime.datetime(2023, 10, 1, 0, 0),
 datetime.datetime(2023, 11, 1, 0, 0),
 datetime.datetime(2023, 12, 1, 0, 0)]

Теперь главный цикл.

In [ ]:
dfs = []                                     # список-накопитель для DataFrame'ов

for month in months:
    date_str = month.strftime('%Y-%m-%d')    # дата → строка: %Y год, %m месяц, %d день
    print(date_str)  # '2023-01-01'          # лог прогресса, чтобы видеть, где мы

    # f-строка: всё внутри {…} вычисляется и подставляется в текст.
    # Сам SQL: обрезаем дату до начала месяца, суммируем цены, группируем по месяцу и категории
    sql = f'''
            select
        	date_trunc('month', order_date):: date as dt_month,
        	category ,
        	sum(price) as total_rev
        from lesson_1.orders o
        where date_trunc('month', order_date):: date = '{date_str}'
        group by dt_month,
        	category
    '''
    print(sql)                               # видно, как {date_str} превратился в конкретную дату

    dfs.append (
        get_from_bd(sql)         # результат месяца кладём в список
    )

    time.sleep(5)                            # ⚠️ пауза 5 секунд × 12 итераций = минута простоя

2023-01-01

            select
        	date_trunc('month', order_date):: date as dt_month,
        	category ,
        	sum(price) as total_rev
        from lesson_1.orders o
        where date_trunc('month', order_date):: date = '2023-01-01'
        group by dt_month,
        	category
    
2023-02-01

            select
        	date_trunc('month', order_date):: date as dt_month,
        	category ,
        	sum(price) as total_rev
        from lesson_1.orders o
        where date_trunc('month', order_date):: date = '2023-02-01'
        group by dt_month,
        	category
    
2023-03-01

            select
        	date_trunc('month', order_date):: date as dt_month,
        	category ,
        	sum(price) as total_rev
        from lesson_1.orders o
        where date_trunc('month', order_date):: date = '2023-03-01'
        group by dt_month,
        	category
    
2023-04-01

            select
        	date_trunc('month', order_date):: date as dt_month,
        	category ,
      

Небольшое отступление про f-строки — механику стоит закрепить отдельно.

In [ ]:
num = 'dfgd'                                                 # в переменной строка, несмотря на имя num
ex_str = f"Вывожу:  date_trunc('month', order_date) {num}"   # префикс f включает подстановку {…}
print(ex_str)                                                # одинарные кавычки внутри двойных экранировать не нужно

Вывожу:  date_trunc('month', order_date) dfgd


<div style="background-color: rgba(244, 194, 194, 0.31); padding: 15px; margin: 15px; border-radius: 8px; border-left: 5px solid rgba(244, 194, 194, 0.57);">

**Что полезно помнить про f-строки**

* Префикс `f` перед кавычками включает подстановку: всё в `{...}` вычисляется.
* Внутри можно не только переменные, но и выражения: `f"{a * b}"`, `f"{round(x, 2)}"`.
* Формат задаётся через двоеточие: `f"{x:.2f}"` — два знака после запятой, `f"{x:,}"` — разделители тысяч.
* Чтобы напечатать **сами фигурные скобки**, их удваивают: `f"{{это не подстановка}}"`. Важно, если в SQL попадается JSON или массив.

Осталось склеить двенадцать кусочков в одну таблицу.

In [ ]:
final_df = pd.concat(dfs)   # склейка «друг под друга» (axis=0 по умолчанию), колонки сопоставляются по именам

final_df

C:\Users\Asus\AppData\Local\Temp\ipykernel_10148\2483640116.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(dfs)   # склейка «друг под друга» (axis=0 по умолчанию), колонки сопоставляются по именам


,dt_month,category,total_rev
0,2023-01-01,American,9284.85
1,2023-01-01,Asian,15588.50
2,2023-01-01,Italian,16727.75
3,2023-01-01,Mexican,12215.85
0,2023-02-01,American,8947.15
1,2023-02-01,Asian,15075.70
2,2023-02-01,Italian,15545.05
3,2023-02-01,Mexican,11222.45
0,2023-03-01,American,10005.75
1,2023-03-01,Asian,16056.45


## SQLite

#### Основные концепции:
- **Таблица** — это набор данных, организованных в строки и столбцы.
- **Столбец** — это отдельное поле данных, характеризующее одно свойство (например, имя, возраст).
- **Строка** — это один экземпляр данных (например, информация об одном человеке).
- **SQL** — это язык запросов, который используется для работы с данными. С его помощью можно писать команды для управления базой данных.

### Библиотека `sqlite3` в Python

Библиотека `sqlite3` позволяет Python взаимодействовать с базой данных SQLite. Она предоставляет следующие ключевые компоненты:
- **Подключение к базе данных**. Это делается через объект `Connection`.
- **Курсор** (`Cursor`) — это объект, через который выполняются SQL-запросы.
- **Запросы** — команды на языке SQL, выполняемые через курсор.

Теперь перейдём к практической части и рассмотрим основные операции с SQLite в Python.

### Подключение к базе данных

Первым шагом является создание или подключение к существующей базе данных. Если база данных не существует, она будет создана автоматически.

Пример:


In [ ]:
import sqlite3

# Подключение к базе данных (создание файла базы данных)
conn = sqlite3.connect('example.db')

# Создание объекта курсора
cur = conn.cursor()

# Проверка успешного подключения
print("Подключение к базе данных установлено!")

Подключение к базе данных установлено!


**Объяснение**:
- `sqlite3.connect('example.db')` — создаёт подключение к файлу базы данных. Если файла нет, он будет создан.
- `conn.cursor()` — создаёт объект курсора для выполнения SQL-запросов.

### Создание таблицы
После подключения к базе данных, можно создавать таблицы. Для этого используется команда `CREATE TABLE`.

Пример создания таблицы:

In [ ]:
cur.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        age INTEGER,
        email TEXT
    )
''')

# Сохранение изменений
conn.commit()

print("Таблица создана успешно!")

Таблица создана успешно!


**Объяснение**:
- `CREATE TABLE IF NOT EXISTS` — создаёт таблицу, если она ещё не существует.
- `id INTEGER PRIMARY KEY AUTOINCREMENT` — столбец `id` будет уникальным идентификатором для каждой строки.
- `name TEXT NOT NULL` — поле `name` должно содержать текст и не может быть пустым.
- `conn.commit()` — сохраняет изменения в базе данных.

### Вставка данных

Чтобы добавить данные в таблицу, используется команда `INSERT INTO`.

Пример:

In [ ]:
cur.execute('''
    INSERT INTO users (name, age, email)
    VALUES ('Alice', 30, 'bob@example.com')
''')

# Сохранение изменений
conn.commit()

In [ ]:
cur.execute('''
    INSERT INTO users (name, age, email)
    VALUES ('Alice', 25, 'alice@example.com')
''')

cur.execute('''
    INSERT INTO users (name, age, email)
    VALUES ('Bob', 30, 'bob@example.com')
''')

# Сохранение изменений
conn.commit()

print("Данные успешно добавлены!")

Данные успешно добавлены!


**Объяснение**:
- `INSERT INTO users (name, age, email)` — вставляет значения в указанные столбцы таблицы `users`.
- `VALUES (...)` — задаёт значения для вставки.

### Выборка данных

Для выборки данных используется команда `SELECT`.

Пример:

In [ ]:
# Выборка всех данных из таблицы
cur.execute('SELECT * FROM users')

# Получение всех строк результата
rows = cur.fetchall()

# Вывод результатов
for row in rows:
    print(row)


(1, 'Alice', 25, 'alice@example.com')
(2, 'Bob', 30, 'bob@example.com')


**Объяснение**:
- `SELECT * FROM users` — выбирает все столбцы из таблицы `users`.
- `cur.fetchall()` — возвращает все строки результата запроса.
- Мы перебираем строки и выводим каждую из них.

###  Обновление данных

Для обновления данных используется команда `UPDATE`.

Пример:

In [ ]:
cur.execute('''
    UPDATE users
    SET age = 26
    WHERE name = 'Alice'
''')

# Сохранение изменений
conn.commit()

print("Данные обновлены!")

Данные обновлены!


**Объяснение**:
- `UPDATE users SET age = 26 WHERE name = 'Alice'` — обновляет возраст для всех пользователей с именем 'Alice'.

###  Удаление данных

Для удаления данных используется команда `DELETE`.

Пример:

In [ ]:
cur.execute('''
    DELETE FROM users
    WHERE name = 'Bob'
''')

# Сохранение изменений
conn.commit()

print("Пользователь удалён!")

Пользователь удалён!


**Объяснение**:
- `DELETE FROM users WHERE name = 'Bob'` — удаляет всех пользователей с именем 'Bob'.

###  Фильтрация данных с использованием `WHERE`

Команда `WHERE` позволяет фильтровать данные, выбирая только те строки, которые соответствуют определенным условиям. Эта команда особенно полезна при работе с большими наборами данных, когда нужно выбрать только определенные записи.

In [ ]:
# Выбираем пользователей старше 25 лет
cur.execute('SELECT * FROM users WHERE age > 25')

# Получаем и выводим результат
rows = cur.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 26, 'alice@example.com')


**Объяснение**:
- `WHERE age > 25` — выбирает всех пользователей, возраст которых больше 25 лет.

Также можно использовать другие операторы для фильтрации:
- `=` — равно.
- `!=` или `<>` — не равно.
- `<`, `>`, `<=`, `>=` — меньше, больше, меньше или равно, больше или равно.

In [ ]:
##Пример с текстовыми полями:

# Выбираем всех пользователей с именем 'Alice'
cur.execute('SELECT * FROM users WHERE name = "Alice"')
rows = cur.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 26, 'alice@example.com')
(3, 'Alice', 30, 'bob@example.com')
(4, 'Alice', 25, 'alice@example.com')


###  Сортировка данных с использованием `ORDER BY`

Команда `ORDER BY` используется для сортировки результатов запроса. Можно отсортировать данные по возрастанию (по умолчанию) или убыванию.

In [ ]:
# Сортировка пользователей по возрасту по возрастанию
cur.execute('SELECT * FROM users ORDER BY age ASC')
rows = cur.fetchall()
for row in rows:
    print(row)

(4, 'Alice', 25, 'alice@example.com')
(1, 'Alice', 26, 'alice@example.com')
(3, 'Alice', 30, 'bob@example.com')
(5, 'Bob', 30, 'bob@example.com')


- `ORDER BY age ASC` — сортирует пользователей по возрасту по возрастанию. Ключевое слово `ASC` означает "по возрастанию".
- Если нужно отсортировать по убыванию, можно использовать `DESC` (descending).

In [ ]:
##Пример сортировки по нескольким столбцам:

# Сортировка сначала по возрасту по возрастанию, а затем по имени по убыванию
cur.execute('SELECT * FROM users ORDER BY age ASC, name DESC')
rows = cur.fetchall()
for row in rows:
    print(row)

(4, 'Alice', 25, 'alice@example.com')
(1, 'Alice', 26, 'alice@example.com')
(5, 'Bob', 30, 'bob@example.com')
(3, 'Alice', 30, 'bob@example.com')


###  Группировка данных с использованием `GROUP BY`

Команда `GROUP BY` используется для группировки данных по определенному полю. Это полезно при работе с агрегатными функциями (например, `COUNT()`, `SUM()`, `AVG()` и др.).

#### Пример группировки с подсчетом:

In [ ]:
cur.execute('SELECT age, COUNT(*) FROM users GROUP BY age')
rows = cur.fetchall()
for row in rows:
    print(row)

(25, 1)
(26, 1)
(30, 2)


**Объяснение**:
- `GROUP BY age` — группирует результаты по возрасту.
- `COUNT(*)` — подсчитывает количество пользователей в каждой группе (по возрасту).

###  Условие для агрегатных функций с `HAVING`

Команда `HAVING` позволяет накладывать условия на группы данных. Она аналогична `WHERE`, но используется после группировки данных.


In [ ]:
# Подсчет количества пользователей для каждого возраста, но вывод только тех возрастов, где более одного пользователя
cur.execute('SELECT age, COUNT(*) FROM users GROUP BY age HAVING COUNT(*) > 1')
rows = cur.fetchall()
for row in rows:
    print(row)

(30, 2)


**Объяснение**:
- `HAVING COUNT(*) > 1` — фильтрует только те группы, в которых больше одного пользователя.

# PandaSQL

In [ ]:
import pandasql as pd_sql
import pandas as pd

In [ ]:
titanic_ds = pd.read_csv("titanic.csv")
titanic_ds.head()
titanic_ds.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [ ]:
pd_sql.sqldf('''
select count(t_ds.sex) total_passengers, t_ds.sex Gender
from titanic_ds as t_ds
group by t_ds.sex
order by t_ds.sex desc
''')

,total_passengers,Gender
0,577,male
1,314,female


In [ ]:
pd_sql.sqldf('''
select t_ds.passengerId,t_ds.pclass,t_ds.name, t_ds.age,
case
    when (t_ds.age > 55 and t_ds.age <= 130) then 'OLD'
    when (t_ds.age <= 55 and t_ds.age > 38) then 'ADULT'
    when (t_ds.age <= 38 and t_ds.age > 22) then 'YOUNG'
    when (t_ds.age <= 22 and t_ds.age >= 15) then 'TEENAGER'
    when t_ds.age <15 then 'CHILD'
    else 'INVALID'
end as 'AGE_GROUP'
from titanic_ds as t_ds
'''
)

,PassengerId,Pclass,Name,Age,AGE_GROUP
0,1,3,"Braund, Mr. Owen Harris",22.0,TEENAGER
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,YOUNG
2,3,3,"Heikkinen, Miss. Laina",26.0,YOUNG
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,YOUNG
4,5,3,"Allen, Mr. William Henry",35.0,YOUNG
...,...,...,...,...,...
886,887,2,"Montvila, Rev. Juozas",27.0,YOUNG
887,888,1,"Graham, Miss. Margaret Edith",19.0,TEENAGER
888,889,3,"Johnston, Miss. Catherine Helen ""Carrie""",NaN,INVALID
889,890,1,"Behr, Mr. Karl Howell",26.0,YOUNG


In [ ]:
pd_sql.sqldf('''
select count(t_ds.sex) Total_Passengers,avg(t_ds.age) Average_Age, t_ds.sex Gender
from titanic_ds as t_ds
group by t_ds.sex
order by t_ds.sex desc
''')

,Total_Passengers,Average_Age,Gender
0,577,30.726645,male
1,314,27.915709,female
